# Install Packages

In [21]:
# pip install git+https://github.com/dnth/rag-datakit.git
# !pip install tiktoken

In [ ]:
# !pip install ipywidgets
# !pip install python-dotenv


  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
Using cached python_dotenv-1.1.1-py3-none-any.whl (20 kB)


# Load ENV

In [1]:
import os
from huggingface_hub import login
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get token from environment
token = os.getenv("HF_TOKEN")
login(token=token)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


# Load SSF Data

In [2]:
from datasets import load_dataset

dataset = load_dataset("dnth/ssf-dataset")
dataset

DatasetDict({
    train: Dataset({
        features: ['Sector', 'Track', 'Job Role', 'Job Role Description', 'Performance Expectation'],
        num_rows: 1885
    })
})

In [3]:
dataset["train"][0]

{'Sector': 'Accountancy',
 'Track': 'Assurance',
 'Job Role': 'Audit Associate / Audit Assistant Associate',
 'Job Role Description': 'The Audit Associate/Audit Assistant Associate undertakes specific stages of audit work under supervision. He/She begins to appreciate the underlying principles behind the tasks assigned to him as part of the audit plan. He is also able to make adjustments to the application of skills to improve the work tasks or solve non-complex issues. The Audit Associate/Audit Assistant Associate operates in a structured work environment. He is able to build relationships, work in a team and identify ethical issues with reference to the code of professional conduct and ethics. He is able to select and apply from a range of known solutions to familiar problems and takes responsibility for his own learning and performance. He is a trustworthy and meticulous individual.',
 'Performance Expectation': 'In accordance with: Singapore Standards on Auditing, Ethics Pronouncem

# SSF Job Description Token Analysis

In [4]:
import tiktoken
from datasets import load_dataset
import numpy as np
import math

# Load the OpenAI API key from environment

text_column = 'Job Role Description'

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

# Get the Job Role Description column
job_descriptions = dataset['train'][text_column]

# Calculate token lengths for all job descriptions
token_lengths = [num_tokens_from_string(description, "cl100k_base") for description in job_descriptions]

# Calculate the average, max, min, and other statistics
average_token_length = np.mean(token_lengths)
max_token_length = np.max(token_lengths)
min_token_length = np.min(token_lengths)
std_dev_token_length = np.std(token_lengths)  # Standard deviation

# Calculate percentiles
p25 = np.percentile(token_lengths, 25)  # 25th percentile
p50 = np.percentile(token_lengths, 50)  # 50th percentile (median)
p75 = np.percentile(token_lengths, 75)  # 75th percentile

# Calculate IQR (Interquartile Range)
IQR = p75 - p25
lower_bound = p25 - 1.5 * IQR
upper_bound = p75 + 1.5 * IQR

# Detect outliers
outliers = [length for length in token_lengths if length < lower_bound or length > upper_bound]

# Print the exact average token length
print(f"Exact average token length: {average_token_length}")

# Round up the average token length to the nearest integer
token_avg_length_rounded = math.ceil(average_token_length)

# Print the rounded-up average token length
print(f"Rounded up average token length: ~{token_avg_length_rounded}")

# Print the max and min token lengths
print(f"Maximum token length: {max_token_length}")
print(f"Minimum token length: {min_token_length}")

# Print the standard deviation and variance
print(f"Standard deviation: {std_dev_token_length}")

# Print percentiles
print(f"25th Percentile: {p25}")
print(f"50th Percentile (Median): {p50}")
print(f"75th Percentile: {p75}")

# Print outliers
print(f"Number of outliers: {len(outliers)}")



Exact average token length: 161.60371352785145
Rounded up average token length: ~162
Maximum token length: 393
Minimum token length: 52
Standard deviation: 49.290052444984674
25th Percentile: 124.0
50th Percentile (Median): 155.0
75th Percentile: 195.0
Number of outliers: 14


# Synthetic Data Generation Setup

In [5]:
import os
from distilabel.models import OpenAILLM, TransformersLLM

# llm = TransformersLLM(
#     model="Qwen/Qwen3-4B-Instruct-2507",
#     device_map="auto",
#     torch_dtype="float16",
# )

llm = OpenAILLM(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
)


In [6]:
context_easy = """
You are an HR assistant tasked with creating job descriptions based on the Singapore SkillsFuture Framework. For each job, create **positive** and **easy negative** variations.

### Positive Descriptions
- Capture the **core responsibilities**, **skills**, and **expectations** of the role.
- Rephrase sentences naturally while keeping the same meaning.
- Start with: "The <job role>."
- Examples:
   - *Positive Example*: "The Operations Manager oversees daily operations and ensures the company's goals are met efficiently."
   - *Rephrased Example*: "The Operations Manager is responsible for managing daily activities and ensuring that the company's objectives are efficiently met."

### Easy Negative Descriptions
- Generate a description that is **clearly unrelated** to the original, with no overlap in skills, sector, or responsibilities.
- The job role should be **completely different** in terms of function, industry, or job focus.
- Example: 
   - *Original Role*: "The Operations Manager ensures smooth day-to-day operations."
   - *Negative Example*: "The Graphic Designer creates digital content for marketing campaigns."

### General Instructions for Positive Descriptions and Easy Negative Descriptions
- Ensure the descriptions are **clear**, **professional**, and **complete**.
- Ensure descriptions are **realistic**, **readable**, and **varied**.
- Avoid **generic** or **incorrect** descriptions.
"""

context_hard = """
You are an HR assistant tasked with creating job descriptions based on the Singapore SkillsFuture Framework. For each job, create **positive** and **hard negative** variations.

### Positive Descriptions
- Capture the **core responsibilities**, **skills**, and **expectations** of the role.
- Rephrase sentences naturally while keeping the same meaning.
- Start with: "The <job role>."
- Examples:
   - *Positive Example*: "The Operations Manager oversees daily operations and ensures the company's goals are met efficiently."
   - *Rephrased Example*: "The Operations Manager is responsible for managing daily activities and ensuring that the company's objectives are efficiently met."

### Hard Negative Descriptions
- Generate a description that is **similar in appearance but semantically different**.
- Strategies:
   - Use a **different seniority level** within the same sector (e.g., Senior Manager → Manager).
   - **Shift the sector** or focus (e.g., Engineering → Operations).
   - **Substitute similar skills in different sectors** (e.g., Finance in Banking → Healthcare).
   - **Similar role, different sector** (e.g., Project Manager in IT → Project Manager in Construction).
- Keep some **similar keywords** to make the negative harder to distinguish.
- Example:
   - *Original Role*: "The Senior Operations Manager oversees high-level operations and strategic implementation."
   - *Hard Negative Example*: "The Senior IT Project Manager manages technology projects, focusing on systems integration."

### General Instructions for Positive Descriptions and Hard Negative Descriptions
- Ensure the descriptions are **clear**, **professional**, and **complete**.
- Ensure descriptions are **realistic**, **readable**, and **varied**.
- Avoid **generic** or **incorrect** descriptions.
"""


In [8]:
from distilabel.pipeline import Pipeline
from distilabel.steps import LoadDataFromHub
from distilabel.steps.tasks import GenerateSentencePair

with Pipeline(name="generate") as pipeline:
    load_dataset = LoadDataFromHub(
        #num_examples=100,  # Limit to 10 examples for demo - increase for production datasets
        use_cache=False,  # Disable caching to ensure fresh data generation each run
        output_mappings={"Job Role Description": "anchor"},  # Map original column to 'anchor' for triplet generation
    )
    generate_retrieval_pairs_easy = GenerateSentencePair(
        name="easy_triplets_paraphrase",
        triplet=True,  # Generate anchor-positive-negative triplets for embedding training
        hard_negative=False,  # Use easier negatives rather than hard negatives
        action="paraphrase",  # Focus on paraphrasing for positive examples
        llm=llm,  # Use the LLM configured above (local Qwen or OpenAI)
        input_batch_size=10,  # Process 10 examples at once for efficiency
        #context=context,  # Provide the context instructions for generation quality
        context=context_easy,  # Provide the context instructions for generation quality
    )
    generate_retrieval_pairs_hard = GenerateSentencePair(
        name="hard_triplets_paraphrase",
        triplet=True,  
        hard_negative=True,  
        action="paraphrase",  
        llm=llm,  
        input_batch_size=10,  
        #context=context,  
        context=context_hard,  
    )
    
    generate_retrieval_pairs_hard_semantic = GenerateSentencePair(
        name="hard_triplets_semantic",
        triplet=True,  # enable negative output
        hard_negative=True,  # negative sentence is crafted to be semantically close to the anchor, making it more challenging for models to distinguish between the positive and negative pairs. This is useful for fine-tuning models to improve their discrimination capabilities.
        action="semantically-similar",  
        llm=llm,  
        input_batch_size=10,  
        #context=context,  
        context=context_hard,  
    )

    load_dataset.connect(generate_retrieval_pairs_easy, generate_retrieval_pairs_hard, generate_retrieval_pairs_hard_semantic)

In [9]:
output_avg_token_length = token_avg_length_rounded*2
output_max_token_length = max_token_length*2

print("Output Token Length:", output_avg_token_length)
print("Output Max Token Length:", output_max_token_length)

Output Token Length: 324
Output Max Token Length: 786


In [10]:
distiset = pipeline.run(
    use_cache=False,
    parameters={
        load_dataset.name: {
            "repo_id": "dnth/ssf-dataset",
            "split": "train",
        },
        "easy_triplets_paraphrase": {
            "llm": {"generation_kwargs": {"temperature": 0.6, "max_new_tokens": output_max_token_length}}
             #"llm": {"generation_kwargs": {"temperature": 0.6}}
        },
        "hard_triplets_paraphrase": {
            "llm": {"generation_kwargs": {"temperature": 0.6, "max_new_tokens": output_max_token_length}}
             #"llm": {"generation_kwargs": {"temperature": 0.6}}
        },
        "hard_triplets_semantic": {
            "llm": {"generation_kwargs": {"temperature": 0.6, "max_new_tokens": output_max_token_length}}
             #"llm": {"generation_kwargs": {"temperature": 0.6}}
        },
    }
)

[08/28/25 15:44:43] INFO     ['distilabel.pipeline'] 📝 Pipeline data will be written to               ]8;id=700126;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=724483;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py#1015\1015]8;;\
                             '/home/frank123/.cache/distilabel/pipelines/generate/4c0dec9e1fa6a01a970c             
                             cd02c94a02229d675e7d/executions/7b585705b0fa70b18bfed7a59d65f9edaffb1c90/             
                             data/steps_outputs'                                                                   

                    INFO     ['distilabel.pipeline'] ⌛ The steps of the pipeline will be loaded in    ]8;id=601485;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=424674;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py#1046\1046]8;;\
                             stages:                                                                               
                              * Legend: 🚰 GeneratorStep 🌐 GlobalStep 🔄 Step                                     
                              * Stage 0:                                                                           
                                - 🚰 'load_data_from_hub_0'                                                        
                                - 🔄 'easy_triplets_paraphrase'                                                    
                                - 🔄 'hard_triplets_paraphrase'                                                    
                                - 🔄 'hard_triplets_semantic'                                                      

                    INFO     ['distilabel.pipeline'] ⏳ Waiting for all the steps of stage 0 to        ]8;id=708587;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=234434;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py#1382\1382]8;;\
                             load...                                                                               

[08/28/25 15:44:45] INFO     ['distilabel.pipeline'] ⏳ Steps from stage 0 loaded: 3/4                 ]8;id=159646;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=852726;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py#1418\1418]8;;\
                              * 'load_data_from_hub_0' replicas: 0/1                                               
                              * 'easy_triplets_paraphrase' replicas: 1/1                                           
                              * 'hard_triplets_paraphrase' replicas: 1/1                                           
                              * 'hard_triplets_semantic' replicas: 1/1                                             

[08/28/25 15:44:48] INFO     ['distilabel.pipeline'] ⏳ Steps from stage 0 loaded: 4/4                 ]8;id=366700;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=967433;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py#1418\1418]8;;\
                              * 'load_data_from_hub_0' replicas: 1/1                                               
                              * 'easy_triplets_paraphrase' replicas: 1/1                                           
                              * 'hard_triplets_paraphrase' replicas: 1/1                                           
                              * 'hard_triplets_semantic' replicas: 1/1                                             

...

Generating train split: 0 examples [00:00, ? examples/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [11]:
distiset

Distiset({
    easy_triplets_paraphrase: DatasetDict({
        train: Dataset({
            features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
            num_rows: 1885
        })
    })
    hard_triplets_semantic: DatasetDict({
        train: Dataset({
            features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
            num_rows: 1885
        })
    })
    hard_triplets_paraphrase: DatasetDict({
        train: Dataset({
            features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
            num_rows: 1885
        })
    })
})

In [12]:
distiset["hard_triplets_semantic"]["train"][-1]

{'Sector': 'Workplace Safety and Health',
 'Track': 'System Audit',
 'Job Role': 'Workplace Safety and Health Auditor',
 'anchor': 'The WSH Auditor is responsible for preparing audit plans, conducting audits and interviews and submitting audit report. He/she is responsible for evaluating an organisations WSH management system, identify areas for improvement, make the relevant recommendations and monitor the progress of improvement. In addition, he is expected to conduct physical inspection of workplace to collect and verify information in accordance to the audit plan. The WSH Auditor is analytical, resourceful, collaborative and has good teamwork.',
 'Performance Expectation': 'In accordance with: Workplace Safety and Health Act',
 'positive': "The WSH Auditor is tasked with developing audit plans, performing audits and interviews, and delivering comprehensive audit reports. He/she evaluates an organization's WSH management system, identifies opportunities for enhancement, provides rel

In [13]:
hard_triplets_semantic_df = distiset["hard_triplets_semantic"]["train"].to_pandas()
hard_triplets_semantic_df

,Sector,Track,Job Role,anchor,Performance Expectation,positive,negative,distilabel_metadata,model_name
0,Accountancy,Assurance,Audit Associate / Audit Assistant Associate,The Audit Associate/Audit Assistant Associate ...,In accordance with: Singapore Standards on Aud...,The Audit Associate/Audit Assistant Associate ...,The Audit Manager oversees various stages of f...,{'raw_input_hard_triplets_semantic': [{'conten...,gpt-4o-mini
1,Accountancy,Assurance,Audit Manager,The Audit Senior Manager/Audit Manager manages...,In accordance with: Singapore Standards on Aud...,The Audit Senior Manager/Audit Manager is resp...,The Audit Manager/Finance Manager oversees a p...,{'raw_input_hard_triplets_semantic': [{'conten...,gpt-4o-mini
2,Accountancy,Assurance,Audit Partner / Audit Director,The Audit Partner/Audit Director is a transfor...,In accordance with: Singapore Standards on Aud...,The Audit Partner/Audit Director is a transfor...,The Audit Manager is a transformational leader...,{'raw_input_hard_triplets_semantic': [{'conten...,gpt-4o-mini
3,Accountancy,Assurance,Audit Senior,The Audit Senior is expected to team lead vari...,In accordance with: Singapore Standards on Aud...,The Audit Senior is responsible for leading va...,The Audit Manager is tasked with coordinating ...,{'raw_input_hard_triplets_semantic': [{'conten...,gpt-4o-mini
4,Accountancy,Business Valuation,Business Valuation Associate / Business Valuat...,The Business Valuation Associate/Business Valu...,In accordance with the International Valuation...,The Business Valuation Associate/Business Valu...,The Business Development Associate/Business De...,{'raw_input_hard_triplets_semantic': [{'conten...,gpt-4o-mini
...,...,...,...,...,...,...,...,...,...
1880,Workplace Safety and Health,Operational Control,Workplace Safety and Health Manager,The WSH Manager is responsible for reviewing W...,In accordance with: Workplace Safety and Healt...,The WSH Manager is tasked with reviewing workp...,The WSH Coordinator is accountable for assessi...,{'raw_input_hard_triplets_semantic': [{'conten...,gpt-4o-mini
1881,Workplace Safety and Health,Operational Control,Workplace Safety and Health Officer,The WSH Officer is responsible for developing ...,In accordance with: Workplace Safety and Healt...,The WSH Officer is tasked with creating and ov...,The WSH Coordinator is in charge of implementi...,{'raw_input_hard_triplets_semantic': [{'conten...,gpt-4o-mini
1882,Workplace Safety and Health,Operational Control,Workplace Safety and Health Supervisor,The Workplace Safety and Health (WSH) Supervis...,In accordance with: Workplace Safety and Healt...,The Workplace Safety and Health (WSH) Supervis...,The Workplace Health and Safety (WHS) Coordina...,{'raw_input_hard_triplets_semantic': [{'conten...,gpt-4o-mini
1883,Workplace Safety and Health,System Audit,Lead Workplace Safety and Health Auditor,The Lead Workplace Safety and Health (WSH) Aud...,In accordance with: Workplace Safety and Healt...,The Lead Workplace Safety and Health (WSH) Aud...,The Lead Quality Assurance (QA) Officer is in ...,{'raw_input_hard_triplets_semantic': [{'conten...,gpt-4o-mini


In [14]:
#distiset.push_to_hub("frankwong2001/ssf-dataset-synthetic_test_2")
distiset.push_to_hub("frankwong2001/ssf-dataset_Full_synthetic_batch10")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  13%|#2        |  528kB / 4.08MB            

...

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  11%|#         |  528kB / 4.92MB            

README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]